# Kamran's letter-image dataset: the current models, trained and tested on it

**Data.** `PSL_Fingerspelling_Image_Dataset/` is the group's letter-image set (the one the published YOLO
models used): full-HD green-screen frames, one folder per letter, split by its authors into train and
validation. Its `Training statistics.doc` gives **2,014 train + 899 validation = 2,913 images**; the
download holds 39 exact duplicate files in train, removed here, which gives exactly those counts.
It is **private**: the folder and the landmark cache inside it are in `.gitignore`; only aggregate
numbers leave it.

**Letters.** 38 letters; no آ ؤ ئ. Kaf is spelled with the Arabic ك in its folder names and is
mapped to the Urdu ک used everywhere else (`image_dataset.LETTER_FIX`).

**Pipeline.** Unchanged: MediaPipe hand landmarks (image mode, same code as the video holds,
`image_dataset.py`) → 74 geometry features (`hand_geometry.py`) → classifier.

**Models.** The shipped SVM (`SVC(C=10, gamma="scale", class_weight="balanced")`, scored by its own
`predict`, as the browser does) and ExtraTrees (`models.extra_trees`, the best single model so far).
Settings are the existing ones; nothing is tuned here.

**Tests.**
* **A.** The models trained on our data only (as shipped: dataset holds + calibration takes), tested on
  Kamran's images. Nothing from his set is used for training.
* **B.** Trained on Kamran's train split, tested on his validation split.
* **C.** Trained on our data + Kamran's train split, tested on his validation split.
* **D.** Does his data help on *our* main question, a new person? Leave-one-signer-out over our five
  dataset signers, with and without all of Kamran's images added to training.

Each is run on all letters and on the 30 static letters (the 11 moving letters are a stated limitation).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, json
import numpy as np, pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import evaluation as ev, models, splits, image_dataset
from hand_geometry import hand_features

OUT = "results/2026-11-20_kamran_images"
os.makedirs(OUT, exist_ok=True)
DYN = set(splits.DYNAMIC_LETTERS)

## 1. Landmarks and clean-up

In [2]:
PK, MK = image_dataset.extract()          # cached after the first run (inside the git-ignored folder)
print("images:", len(MK))
print(pd.crosstab(MK.split, MK.found_in, margins=True))

# exact duplicate files: keep the first copy
dup = MK.duplicated("sha1")
print("\nexact duplicates dropped:", int(dup.sum()), dict(MK[dup].split.value_counts()))
print("files in both train and val:", len(set(MK.sha1[MK.split == "train"]) & set(MK.sha1[MK.split == "val"])))
keep = ~dup & (MK.detected == 1)
PK, MK = PK[keep.to_numpy()], MK[keep].reset_index(drop=True)
print("\nusable (hand found, no duplicates):", dict(MK.split.value_counts()))
print("hand reported:", dict(MK.hand.value_counts()))

images: 2952
found_in  centre  full  none   All
split                             
train         88  1959     6  2053
val           42   852     5   899
All          130  2811    11  2952

exact duplicates dropped: 39 {'train': np.int64(39)}
files in both train and val: 0

usable (hand found, no duplicates): {'train': np.int64(2008), 'val': np.int64(894)}
hand reported: {'Right': np.int64(1485), 'Left': np.int64(1417)}


In [3]:
P_all, M_all = ev.load_dataset(include_users=True)     # our holds, as shipped
user = splits.is_user(M_all.signer)
print("our holds:", int((~user).sum()), "dataset +", int(user.sum()), "calibration;",
      M_all.letter.nunique(), "letters")
FK, yK, sK = hand_features(PK), MK.letter.to_numpy(), MK.split.to_numpy()
F_all, y_all, g_all = hand_features(P_all), M_all.letter.to_numpy(), M_all.signer.to_numpy()
print("letters in Kamran's set:", len(set(yK)), "| in ours, not his:", sorted(set(y_all) - set(yK)))

our holds: 1345 dataset + 291 calibration; 41 letters
letters in Kamran's set: 38 | in ours, not his: ['آ', 'ؤ', 'ئ']


## 2. Overlap checks

Two things would make the scores below too good:
1. **His validation images are near copies of his training images** (frames from the same video split
   at random). Then B and C measure memory, not generalisation.
2. **His images contain our signers' frames.** Then D would leak the held-out signer into training.

Both are checked with nearest-neighbour distances in the standardised 74-feature space, compared
with two yardsticks from our own data: the nearest hold of the **same** signer and of a **different** signer
(same letter in both cases).

In [4]:
sc = StandardScaler().fit(np.vstack([F_all[~user], FK]))
Z, ZK = sc.transform(F_all[~user]), sc.transform(FK)
yD, gD = y_all[~user], g_all[~user]

def nn_dist(Zq, yq, Zr, yr, mask=None):
    """distance from each query row to the nearest reference row of the same letter"""
    out = np.full(len(Zq), np.nan)
    for L in set(yq):
        q = np.where(yq == L)[0]
        for i in q:
            r = (yr == L) if mask is None else (yr == L) & mask(i)
            if r.any():
                out[i] = np.min(np.linalg.norm(Zr[r] - Zq[i], axis=1))
    return out

same = nn_dist(Z, yD, Z, yD, mask=lambda i: (gD == gD[i]) & (np.arange(len(gD)) != i))
other = nn_dist(Z, yD, Z, yD, mask=lambda i: gD != gD[i])
tr, va = sK == "train", sK == "val"
k_val_to_train = nn_dist(ZK[va], yK[va], ZK[tr], yK[tr])
ours_to_k = nn_dist(Z, yD, ZK, yK)

ov = pd.DataFrame({
    "our hold -> nearest hold, same signer": same,
    "our hold -> nearest hold, other signer": pd.Series(other),
    "Kamran val -> nearest Kamran train": pd.Series(k_val_to_train),
    "our hold -> nearest Kamran image": pd.Series(ours_to_k),
}).describe(percentiles=[.1, .5]).T[["count", "10%", "50%", "mean"]].round(2)
ov

,count,10%,50%,mean
"our hold -> nearest hold, same signer",1303.0,1.55,4.14,5.06
"our hold -> nearest hold, other signer",1345.0,2.65,4.49,4.96
Kamran val -> nearest Kamran train,894.0,0.28,0.65,1.08
our hold -> nearest Kamran image,1303.0,2.46,5.88,6.09


In [5]:
thr = np.nanpercentile(same, 10)
print(f"share closer than the 10th percentile of same-signer distances ({thr:.2f}):")
print(f"  Kamran val  -> Kamran train : {np.nanmean(k_val_to_train < thr):.1%}")
print(f"  our holds   -> Kamran images: {np.nanmean(ours_to_k < thr):.1%}")
print(f"  our holds   -> other signer : {np.nanmean(other < thr):.1%}   (yardstick)")

share closer than the 10th percentile of same-signer distances (1.55):
  Kamran val  -> Kamran train : 84.1%
  our holds   -> Kamran images: 2.8%
  our holds   -> other signer : 0.1%   (yardstick)


## 3. Scores

In [6]:
MODELS = {"SVM (shipped)": lambda: ev.svm(), "ExtraTrees": lambda: models.extra_trees(0)}

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return 100*(c-h), 100*(c+h)

def score(Ftr, ytr, Fte, yte):
    rows, preds = [], {}
    for name, make in MODELS.items():
        pred = make().fit(Ftr, ytr).predict(Fte)
        k = int((pred == yte).sum()); lo, hi = wilson(k, len(yte))
        rows.append({"model": name, "n test": len(yte), "accuracy %": 100*k/len(yte),
                     "95% CI": f"{lo:.1f}-{hi:.1f}",
                     "macro recall %": 100*recall_score(yte, pred, average="macro", labels=sorted(set(yte)))})
        preds[name] = pred
    return rows, preds

def letters(y, static):
    return ~np.isin(y, list(DYN)) if static else np.ones(len(y), bool)

results, PRED = [], {}
for static in (False, True):
    tag = "30 static letters" if static else "all letters"
    a, k = letters(y_all, static), letters(yK, static)
    kt, kv = k & tr, k & va
    tests = {
        "A. ours only -> Kamran val":            (F_all[a], y_all[a], FK[kv], yK[kv]),
        "A. ours only -> all Kamran images":     (F_all[a], y_all[a], FK[k], yK[k]),
        "B. Kamran train -> Kamran val":         (FK[kt], yK[kt], FK[kv], yK[kv]),
        "C. ours + Kamran train -> Kamran val":  (np.vstack([F_all[a], FK[kt]]), np.r_[y_all[a], yK[kt]], FK[kv], yK[kv]),
    }
    for test, (Ftr, ytr, Fte, yte) in tests.items():
        rows, preds = score(Ftr, ytr, Fte, yte)
        PRED[(tag, test)] = (yte, preds)
        for r in rows:
            results.append({"letters": tag, "test": test, "n train": len(ytr), **r})

R = pd.DataFrame(results)
R.round(1)

,letters,test,n train,model,n test,accuracy %,95% CI,macro recall %
0,all letters,A. ours only -> Kamran val,1636,SVM (shipped),894,30.0,27.1-33.1,30.8
1,all letters,A. ours only -> Kamran val,1636,ExtraTrees,894,35.9,32.8-39.1,36.7
2,all letters,A. ours only -> all Kamran images,1636,SVM (shipped),2902,29.3,27.6-30.9,30.6
3,all letters,A. ours only -> all Kamran images,1636,ExtraTrees,2902,35.2,33.5-37.0,36.8
4,all letters,B. Kamran train -> Kamran val,2008,SVM (shipped),894,84.3,81.8-86.6,84.1
5,all letters,B. Kamran train -> Kamran val,2008,ExtraTrees,894,93.6,91.8-95.0,93.7
6,all letters,C. ours + Kamran train -> Kamran val,3644,SVM (shipped),894,81.4,78.8-83.8,81.3
7,all letters,C. ours + Kamran train -> Kamran val,3644,ExtraTrees,894,92.8,91.0-94.4,92.7
8,30 static letters,A. ours only -> Kamran val,1319,SVM (shipped),726,34.3,30.9-37.8,35.3
9,30 static letters,A. ours only -> Kamran val,1319,ExtraTrees,726,39.7,36.2-43.3,40.7


### D. A new person, with and without Kamran's images in training

Leave-one-signer-out over our five dataset signers (calibration takes left out, as in the earlier
new-person results). The test is always one of *our* signers; the only change is whether all of
Kamran's images join the training rows.

In [7]:
rowsD = []
for static in (False, True):
    tag = "30 static letters" if static else "all letters"
    a, k = letters(yD, static), letters(yK, static)
    for name, make in MODELS.items():
        for s in sorted(set(gD)):
            te, trn = a & (gD == s), a & (gD != s)
            for added in (False, True):
                Ftr = np.vstack([F_all[~user][trn], FK[k]]) if added else F_all[~user][trn]
                ytr = np.r_[yD[trn], yK[k]] if added else yD[trn]
                pred = make().fit(Ftr, ytr).predict(F_all[~user][te])
                rowsD.append({"letters": tag, "model": name, "test signer": s,
                              "Kamran added": added, "correct": int((pred == yD[te]).sum()), "n": int(te.sum()),
                              "macro recall": recall_score(yD[te], pred, average="macro", labels=sorted(set(yD[te])))})
D = pd.DataFrame(rowsD)
D["accuracy %"] = 100 * D.correct / D.n

per = D.pivot_table(index=["letters", "model", "test signer"], columns="Kamran added", values="accuracy %").round(1)
per.columns = ["ours only", "+ Kamran"]
per["change"] = per["+ Kamran"] - per["ours only"]
per

ours only  + Kamran  change
letters           model         test signer                             
30 static letters ExtraTrees    signer1-F         34.6      36.8     2.2
                                signer2-F         43.2      44.1     0.9
                                signer3-F         40.1      41.0     0.9
                                signer4-L         37.5      43.8     6.3
                                signer5-R         42.7      46.8     4.1
                  SVM (shipped) signer1-F         34.6      37.2     2.6
                                signer2-F         37.6      41.9     4.3
                                signer3-F         35.5      38.7     3.2
                                signer4-L         35.3      42.9     7.6
                                signer5-R         38.7      47.6     8.9
all letters       ExtraTrees    signer1-F         28.3      28.9     0.6
                                signer2-F         39.1      35.4    -3.7
                                signer3-F         33.7      32.6    -1.1
                                signer4-L         26.8      36.9    10.1
                                signer5-R         31.0      36.1     5.1
                  SVM (shipped) signer1-F         21.9      28.6     6.7
                                signer2-F         32.5      34.1     1.6
                                signer3-F         29.0      31.5     2.5
                                signer4-L         22.0      32.9    10.9
                                signer5-R         27.2      38.6    11.4

In [8]:
summ = []
for (tag, name), d in D.groupby(["letters", "model"]):
    b, w = d[~d["Kamran added"]].sort_values("test signer"), d[d["Kamran added"]].sort_values("test signer")
    pooled = lambda x: 100 * x.correct.sum() / x.n.sum()
    diff = w["accuracy %"].to_numpy() - b["accuracy %"].to_numpy()
    summ.append({"letters": tag, "model": name,
                 "ours only %": b["accuracy %"].mean(), "+ Kamran %": w["accuracy %"].mean(),
                 "change (pts)": diff.mean(), "signers improved": f"{(diff > 0).sum()} of {len(diff)}",
                 "pooled ours only %": pooled(b), "pooled + Kamran %": pooled(w),
                 "macro recall ours only %": 100*b["macro recall"].mean(),
                 "macro recall + Kamran %": 100*w["macro recall"].mean()})
S = pd.DataFrame(summ)
S.round(1)

,letters,model,ours only %,+ Kamran %,change (pts),signers improved,pooled ours only %,pooled + Kamran %,macro recall ours only %,macro recall + Kamran %
0,30 static letters,ExtraTrees,39.6,42.5,2.8,5 of 5,39.3,42.0,33.7,42.0
1,30 static letters,SVM (shipped),36.3,41.6,5.3,5 of 5,36.1,41.1,30.6,40.9
2,all letters,ExtraTrees,31.8,34.0,2.2,3 of 5,31.8,33.8,27.5,35.0
3,all letters,SVM (shipped),26.5,33.2,6.6,5 of 5,26.4,32.6,24.0,33.2


## 4. Which letters fail on Kamran's images (test A, ExtraTrees, all letters)

In [9]:
yte, preds = PRED[("all letters", "A. ours only -> all Kamran images")]
p = preds["ExtraTrees"]
L = pd.DataFrame({"letter": yte, "ok": p == yte, "pred": p})
per_letter = L.groupby("letter").agg(n=("ok", "size"), recall=("ok", "mean"),
                                     most_common_guess=("pred", lambda s: s.value_counts().index[0]))
per_letter["recall"] = (100 * per_letter.recall).round(0)
per_letter.sort_values("recall")

,n,recall,most_common_guess
letter,,,
ہ,112,2.0,ک
گ,90,12.0,ز
پ,54,15.0,س
ط,79,15.0,ث
م,63,19.0,ث
ذ,100,19.0,ر
ء,90,19.0,ئ
ل,98,20.0,و
ک,80,22.0,ث


In [10]:
R.to_csv(f"{OUT}/scores_ABC.csv", index=False, encoding="utf-8")
D.to_csv(f"{OUT}/scores_D_per_signer.csv", index=False, encoding="utf-8")
S.to_csv(f"{OUT}/scores_D_summary.csv", index=False, encoding="utf-8")
ov.to_csv(f"{OUT}/overlap_checks.csv", encoding="utf-8")
print("written to", OUT)

written to results/2026-11-20_kamran_images


## 5. Which model should the browser use, now that Kamran's images are in training?

Test D again (new person, our five signers held out one at a time, all of Kamran's images in
training), for the shipped SVM, ExtraTrees, and the 11 Nov ensemble (mean probability of
SVM + MLP + ExtraTrees + LDA, `models.ensemble`), the model that gave the best **word** accuracy
then (81.4% against the SVM's 74.8%, `model_alternatives.ipynb`). Settings are the existing ones.
The ensemble and SVM here are scored by `argmax(predict_proba)`; the SVM row is also given by its
own `predict`, which is what the browser uses.

In [11]:
CAND = {
    "SVM (shipped), predict": lambda: ev.svm(),
    "SVM, argmax probability": lambda: models.svm(0),
    "ExtraTrees": lambda: models.extra_trees(0),
    "ensemble SVM+MLP+ET+LDA": lambda: models.ensemble(0),
}
rows5 = []
for static in (False, True):
    tag = "30 static letters" if static else "all letters"
    a, k = letters(yD, static), letters(yK, static)
    for s in sorted(set(gD)):
        te, trn = a & (gD == s), a & (gD != s)
        Ftr = np.vstack([F_all[~user][trn], FK[k]]); ytr = np.r_[yD[trn], yK[k]]
        for name, make in CAND.items():
            m = make().fit(Ftr, ytr)
            pred = m.predict(F_all[~user][te]) if "predict" in name else m.classes_[m.predict_proba(F_all[~user][te]).argmax(1)]
            rows5.append({"letters": tag, "model": name, "test signer": s, "accuracy %": 100*(pred == yD[te]).mean(),
                          "macro recall %": 100*recall_score(yD[te], pred, average="macro", labels=sorted(set(yD[te])))})
E = pd.DataFrame(rows5)
tab = E.groupby(["letters", "model"])[["accuracy %", "macro recall %"]].mean().round(1)
base = E[E.model == "SVM (shipped), predict"].set_index(["letters", "test signer"])["accuracy %"]
for (tag, name), d in E.groupby(["letters", "model"]):
    diff = d.set_index(["letters", "test signer"])["accuracy %"] - base.loc[tag].reindex(d["test signer"]).set_axis(d.set_index(["letters", "test signer"]).index)
    tab.loc[(tag, name), "vs shipped SVM (pts)"] = round(diff.mean(), 1)
    tab.loc[(tag, name), "signers better"] = f"{(diff > 0).sum()} of {len(diff)}"
E.to_csv(f"{OUT}/model_choice_with_kamran.csv", index=False, encoding="utf-8")
tab

accuracy %  macro recall %  \
letters           model                                                 
30 static letters ExtraTrees                     42.5            42.0   
                  SVM (shipped), predict         41.6            40.9   
                  SVM, argmax probability        42.0            36.9   
                  ensemble SVM+MLP+ET+LDA        43.1            43.1   
all letters       ExtraTrees                     34.0            35.0   
                  SVM (shipped), predict         33.2            33.2   
                  SVM, argmax probability        35.2            31.5   
                  ensemble SVM+MLP+ET+LDA        34.3            34.5   

                                           vs shipped SVM (pts) signers better  
letters           model                                                         
30 static letters ExtraTrees                                0.8         3 of 5  
                  SVM (shipped), predict                    0.0         0 of 5  
                  SVM, argmax probability                   0.4         4 of 5  
                  ensemble SVM+MLP+ET+LDA                   1.4         3 of 5  
all letters       ExtraTrees                                0.9         4 of 5  
                  SVM (shipped), predict                    0.0         0 of 5  
                  SVM, argmax probability                   2.0         4 of 5  
                  ensemble SVM+MLP+ET+LDA                   1.1         4 of 5